# ANÁLISIS DE DATOS DEL NEGOCIO - GOOGLE COLAB




# Este notebook analiza los datos de ventas, inventario, clientes y finanzas del negocio utilizando 10 tablas relacionadas.

## 1. Instalación y Configuración

In [1]:
# Instalamos las librerías necesarias
!pip install -q plotly kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.3/66.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.6 MB/s eta 0:00:00


In [2]:
# Importamos las librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'${x:,.2f}' if x > 100 else f'{x:.2f}')

print("✓ Librerías importadas correctamente")

✓ Librerías importadas correctamente


## 2. Carga de Datos

In [4]:
from google.colab import files

print("Por favor, sube los 10 archivos CSV:")
print("- clientes.csv")
print("- empleados.csv")
print("- proveedores.csv")
print("- categorias.csv")
print("- productos.csv")
print("- facturas.csv")
print("- detalle_facturas.csv")
print("- pagos.csv")
print("- compras.csv")
print("- detalle_compras.csv")

uploaded = files.upload()


Por favor, sube los 10 archivos CSV:
- clientes.csv
- empleados.csv
- proveedores.csv
- categorias.csv
- productos.csv
- facturas.csv
- detalle_facturas.csv
- pagos.csv
- compras.csv
- detalle_compras.csv


Saving categorias.csv to categorias.csv
Saving clientes.csv to clientes.csv
Saving compras.csv to compras.csv
Saving detalle_compras.csv to detalle_compras.csv
Saving detalle_facturas.csv to detalle_facturas.csv
Saving empleados.csv to empleados.csv
Saving facturas.csv to facturas.csv
Saving pagos.csv to pagos.csv
Saving productos.csv to productos.csv
Saving proveedores.csv to proveedores.csv


In [5]:
# Cargamos todos los CSVs
print("\n" + "="*60)
print("CARGANDO DATOS...")
print("="*60 + "\n")

try:
    clientes = pd.read_csv('clientes.csv')
    empleados = pd.read_csv('empleados.csv')
    proveedores = pd.read_csv('proveedores.csv')
    categorias = pd.read_csv('categorias.csv')
    productos = pd.read_csv('productos.csv')
    facturas = pd.read_csv('facturas.csv')
    detalle_facturas = pd.read_csv('detalle_facturas.csv')
    pagos = pd.read_csv('pagos.csv')
    compras = pd.read_csv('compras.csv')
    detalle_compras = pd.read_csv('detalle_compras.csv')

    tablas = {
        'Clientes': clientes,
        'Empleados': empleados,
        'Proveedores': proveedores,
        'Categorías': categorias,
        'Productos': productos,
        'Facturas': facturas,
        'Detalle Facturas': detalle_facturas,
        'Pagos': pagos,
        'Compras': compras,
        'Detalle Compras': detalle_compras
    }

    for nombre, df in tablas.items():
        print(f"✓ {nombre}: {len(df)} registros, {len(df.columns)} columnas")

    print("\n✓ Todos los archivos cargados exitosamente")

except Exception as e:
    print(f"✗ Error al cargar archivos: {e}")
    print("Asegúrate de haber subido todos los archivos CSV")


CARGANDO DATOS...

✓ Clientes: 50 registros, 10 columnas
✓ Empleados: 15 registros, 10 columnas
✓ Proveedores: 11 registros, 10 columnas
✓ Categorías: 7 registros, 4 columnas
✓ Productos: 30 registros, 12 columnas
✓ Facturas: 100 registros, 13 columnas
✓ Detalle Facturas: 429 registros, 7 columnas
✓ Pagos: 61 registros, 7 columnas
✓ Compras: 40 registros, 10 columnas
✓ Detalle Compras: 126 registros, 6 columnas

✓ Todos los archivos cargados exitosamente


## 3. Limpieza y Preparación de Datos

In [6]:
print("\n" + "="*60)
print("LIMPIEZA Y PREPARACIÓN DE DATOS")
print("="*60 + "\n")

# Convertir columnas de fecha
columnas_fecha = {
    'clientes': ['fecha_registro'],
    'empleados': ['fecha_contratacion'],
    'facturas': ['fecha_emision', 'fecha_vencimiento'],
    'pagos': ['fecha_pago'],
    'compras': ['fecha_orden', 'fecha_entrega']
}

for tabla, columnas in columnas_fecha.items():
    df = eval(tabla)
    for col in columnas:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])
            print(f"✓ {tabla}.{col} convertido a datetime")


LIMPIEZA Y PREPARACIÓN DE DATOS

✓ clientes.fecha_registro convertido a datetime
✓ empleados.fecha_contratacion convertido a datetime
✓ facturas.fecha_emision convertido a datetime
✓ facturas.fecha_vencimiento convertido a datetime
✓ pagos.fecha_pago convertido a datetime
✓ compras.fecha_orden convertido a datetime
✓ compras.fecha_entrega convertido a datetime


In [7]:
# Convertir columnas numéricas
columnas_numericas = {
    'clientes': ['puntos_fidelidad'],
    'empleados': ['salario'],
    'proveedores': ['calificacion'],
    'categorias': ['comision_vendedor'],
    'productos': ['precio_compra', 'precio_venta', 'stock_actual', 'stock_minimo'],
    'facturas': ['subtotal', 'descuento', 'impuestos', 'total'],
    'detalle_facturas': ['cantidad', 'precio_unitario', 'descuento', 'subtotal'],
    'pagos': ['monto'],
    'compras': ['total'],
    'detalle_compras': ['cantidad', 'precio_unitario', 'subtotal']
}

for tabla, columnas in columnas_numericas.items():
    df = eval(tabla)
    for col in columnas:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

print("\n✓ Datos preparados correctamente")


✓ Datos preparados correctamente


## 4. Información General de las Tablas

In [8]:
print("\n" + "="*60)
print("INFORMACIÓN GENERAL")
print("="*60 + "\n")

for nombre, df in tablas.items():
    print(f"\n{nombre}:")
    print(f"  Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
    print(f"  Valores nulos: {df.isnull().sum().sum()}")
    print(f"  Memoria: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")


INFORMACIÓN GENERAL


Clientes:
  Dimensiones: 50 filas × 10 columnas
  Valores nulos: 0
  Memoria: 20.24 KB

Empleados:
  Dimensiones: 15 filas × 10 columnas
  Valores nulos: 0
  Memoria: 6.97 KB

Proveedores:
  Dimensiones: 11 filas × 10 columnas
  Valores nulos: 0
  Memoria: 6.00 KB

Categorías:
  Dimensiones: 7 filas × 4 columnas
  Valores nulos: 0
  Memoria: 1.13 KB

Productos:
  Dimensiones: 30 filas × 12 columnas
  Valores nulos: 0
  Memoria: 9.89 KB

Facturas:
  Dimensiones: 100 filas × 13 columnas
  Valores nulos: 59
  Memoria: 30.15 KB

Detalle Facturas:
  Dimensiones: 429 filas × 7 columnas
  Valores nulos: 0
  Memoria: 23.59 KB

Pagos:
  Dimensiones: 61 filas × 7 columnas
  Valores nulos: 27
  Memoria: 12.50 KB

Compras:
  Dimensiones: 40 filas × 10 columnas
  Valores nulos: 19
  Memoria: 10.93 KB

Detalle Compras:
  Dimensiones: 126 filas × 6 columnas
  Valores nulos: 0
  Memoria: 6.04 KB


## 5. Análisis Exploratorio de Datos (EDA)

### 5.1 Distribución de Clientes por Ciudad

In [9]:
fig = px.bar(clientes['ciudad'].value_counts().reset_index(),
             x='ciudad', y='count',
             title='Distribución de Clientes por Ciudad',
             labels={'ciudad': 'Ciudad', 'count': 'Número de Clientes'},
             color='count',
             color_continuous_scale='Blues')
fig.update_layout(showlegend=False, height=500)
fig.show()

print(f"\n📍 Total de ciudades: {clientes['ciudad'].nunique()}")
print(f"📍 Ciudad con más clientes: {clientes['ciudad'].value_counts().index[0]} ({clientes['ciudad'].value_counts().values[0]} clientes)")


📍 Total de ciudades: 12
📍 Ciudad con más clientes: Córdoba (7 clientes)


### 5.2 Productos por Categoría

In [10]:
productos_cat = productos.merge(categorias, left_on='id_categoria', right_on='id', how='left')

fig = px.pie(productos_cat, names='nombre_y',
             title='Distribución de Productos por Categoría',
             hole=0.4)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

print(f"\n📦 Total de productos: {len(productos)}")
print(f"📦 Total de categorías: {len(categorias)}")
print(f"📦 Productos activos: {len(productos[productos['activo'] == 'Sí'])}")
print(f"📦 Productos con stock bajo: {len(productos[productos['stock_actual'] < productos['stock_minimo']])}")


📦 Total de productos: 30
📦 Total de categorías: 7
📦 Productos activos: 25
📦 Productos con stock bajo: 2


### 5.3 Estados de Facturas

In [12]:
fig = px.pie(facturas, names='estado',
             title='Distribución de Estados de Facturas',
             color_discrete_sequence=px.colors.qualitative.Set3)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

print("\n📄 Resumen de Facturas:")
print(facturas['estado'].value_counts())


📄 Resumen de Facturas:
estado
Cancelada    28
Pagada       26
Vencida      24
Pendiente    22
Name: count, dtype: int64


## 6. Análisis de Ventas

### 6.1 Ventas Totales por Mes

In [13]:
facturas['mes'] = facturas['fecha_emision'].dt.to_period('M')
ventas_mes = facturas.groupby('mes')['total'].sum().reset_index()
ventas_mes['mes'] = ventas_mes['mes'].astype(str)

fig = px.line(ventas_mes, x='mes', y='total',
              title='Evolución de Ventas Mensuales',
              labels={'mes': 'Mes', 'total': 'Total de Ventas (ARS)'},
              markers=True)
fig.update_traces(line_color='#2E86AB', line_width=3)
fig.update_layout(hovermode='x unified', height=500)
fig.show()

print(f"\n💰 Total de ventas: ${facturas['total'].sum():,.2f}")
print(f"💰 Promedio de ventas por factura: ${facturas['total'].mean():,.2f}")
print(f"💰 Factura más alta: ${facturas['total'].max():,.2f}")
print(f"💰 Factura más baja: ${facturas['total'].min():,.2f}")


💰 Total de ventas: $2,858,614.88
💰 Promedio de ventas por factura: $28,586.15
💰 Factura más alta: $60,493.61
💰 Factura más baja: $901.32


### 6.2 Top 10 Productos Más Vendidos

In [14]:
# Unimos detalle_facturas con productos
ventas_productos = detalle_facturas.merge(productos, left_on='id_producto', right_on='id', how='left')
top_productos = ventas_productos.groupby('nombre')['cantidad'].sum().sort_values(ascending=False).head(10)

fig = px.bar(top_productos.reset_index(),
             x='cantidad', y='nombre',
             orientation='h',
             title='Top 10 Productos Más Vendidos',
             labels={'cantidad': 'Unidades Vendidas', 'nombre': 'Producto'},
             color='cantidad',
             color_continuous_scale='Viridis')
fig.update_layout(showlegend=False, height=500, yaxis={'categoryorder': 'total ascending'})
fig.show()

print("\n🏆 Top 5 Productos Más Vendidos:")
for i, (producto, cant) in enumerate(top_productos.head().items(), 1):
    print(f"{i}. {producto}: {int(cant)} unidades")


🏆 Top 5 Productos Más Vendidos:
1. Producto 27: 129 unidades
2. Producto 2: 112 unidades
3. Producto 23: 110 unidades
4. Producto 5: 106 unidades
5. Producto 26: 105 unidades


### 6.3 Top 10 Clientes que Más Compran

In [15]:
ventas_clientes = facturas.merge(clientes, left_on='id_cliente', right_on='id', how='left')
top_clientes = ventas_clientes.groupby(['nombre', 'apellido'])['total'].sum().sort_values(ascending=False).head(10)
top_clientes.index = [f"{nombre} {apellido}" for nombre, apellido in top_clientes.index]

fig = px.bar(top_clientes.reset_index(),
             x='total', y='index',
             orientation='h',
             title='Top 10 Clientes por Volumen de Compras',
             labels={'total': 'Total Comprado (ARS)', 'index': 'Cliente'},
             color='total',
             color_continuous_scale='Sunset')
fig.update_layout(showlegend=False, height=500, yaxis={'categoryorder': 'total ascending'})
fig.show()

print("\n👥 Top 5 Clientes:")
for i, (cliente, total) in enumerate(top_clientes.head().items(), 1):
    print(f"{i}. {cliente}: ${total:,.2f}")



👥 Top 5 Clientes:
1. Laura Romero: $231,215.94
2. María Flores: $186,631.84
3. Carlos Vargas: $184,757.77
4. Nicolás Castro: $141,335.61
5. Laura Pérez: $134,287.28


### 6.4 Ventas por Método de Pago

In [16]:
ventas_pago = facturas.groupby('tipo_pago')['total'].agg(['sum', 'count']).reset_index()
ventas_pago.columns = ['Método de Pago', 'Total', 'Cantidad']

fig = make_subplots(rows=1, cols=2,
                    specs=[[{'type': 'bar'}, {'type': 'pie'}]],
                    subplot_titles=('Monto por Método de Pago', 'Cantidad de Transacciones'))

fig.add_trace(go.Bar(x=ventas_pago['Método de Pago'], y=ventas_pago['Total'],
                     marker_color='lightblue', name='Total'),
              row=1, col=1)

fig.add_trace(go.Pie(labels=ventas_pago['Método de Pago'], values=ventas_pago['Cantidad'],
                     name='Cantidad'),
              row=1, col=2)

fig.update_layout(title_text='Análisis de Métodos de Pago', height=500)
fig.show()

print("\n💳 Métodos de Pago:")
for _, row in ventas_pago.iterrows():
    print(f"{row['Método de Pago']}: ${row['Total']:,.2f} ({int(row['Cantidad'])} transacciones)")


💳 Métodos de Pago:
Efectivo: $706,359.65 (25 transacciones)
Tarjeta Crédito: $745,446.04 (24 transacciones)
Tarjeta Débito: $745,833.67 (26 transacciones)
Transferencia: $660,975.52 (25 transacciones)


### 6.5 Ticket Promedio por Cliente

In [17]:
ticket_promedio = facturas.groupby('id_cliente')['total'].mean().reset_index()
ticket_promedio = ticket_promedio.merge(clientes[['id', 'nombre', 'apellido']],
                                        left_on='id_cliente', right_on='id', how='left')

print(f"\n🎫 Ticket Promedio General: ${facturas['total'].mean():,.2f}")
print(f"🎫 Mediana de Ticket: ${facturas['total'].median():,.2f}")

fig = px.histogram(ticket_promedio, x='total',
                   title='Distribución de Ticket Promedio por Cliente',
                   labels={'total': 'Ticket Promedio (ARS)', 'count': 'Número de Clientes'},
                   nbins=30)
fig.update_layout(height=500)
fig.show()


🎫 Ticket Promedio General: $28,586.15
🎫 Mediana de Ticket: $27,697.67


# 7. Análisis de Inventario

### 7.1 Productos con Stock Bajo

In [18]:
stock_bajo = productos[productos['stock_actual'] < productos['stock_minimo']].copy()
stock_bajo = stock_bajo.merge(categorias, left_on='id_categoria', right_on='id', how='left')

if len(stock_bajo) > 0:
    print(f"\n⚠️ ALERTA: {len(stock_bajo)} productos con stock bajo del mínimo\n")
    print(stock_bajo[['nombre_x', 'nombre_y', 'stock_actual', 'stock_minimo']].to_string(index=False))

    fig = px.bar(stock_bajo.head(10), x='nombre_x', y=['stock_actual', 'stock_minimo'],
                 title='Productos con Stock Bajo (Top 10)',
                 labels={'value': 'Cantidad', 'nombre_x': 'Producto'},
                 barmode='group')
    fig.update_layout(height=500)
    fig.show()
else:
    print("\n✓ Todos los productos tienen stock suficiente")


⚠️ ALERTA: 2 productos con stock bajo del mínimo

   nombre_x nombre_y  stock_actual  stock_minimo
 Producto 3    Hogar            10            11
Producto 28    Hogar             4             7


### 7.2 Productos Más Rentables

In [19]:
productos['margen'] = ((productos['precio_venta'] - productos['precio_compra']) / productos['precio_compra'] * 100)
productos['ganancia_unitaria'] = productos['precio_venta'] - productos['precio_compra']

top_rentables = productos.nlargest(10, 'margen')[['nombre', 'precio_compra', 'precio_venta', 'margen', 'ganancia_unitaria']]

fig = px.bar(top_rentables, x='nombre', y='margen',
             title='Top 10 Productos Más Rentables (% Margen)',
             labels={'margen': 'Margen de Ganancia (%)', 'nombre': 'Producto'},
             color='margen',
             color_continuous_scale='Greens')
fig.update_layout(showlegend=False, height=500)
fig.show()

print("\n💎 Top 5 Productos Más Rentables:")
for i, row in top_rentables.head().iterrows():
    print(f"{row['nombre']}: {row['margen']:.1f}% margen (${row['ganancia_unitaria']:,.2f} por unidad)")



💎 Top 5 Productos Más Rentables:
Producto 28: 147.9% margen ($1,320.02 por unidad)
Producto 29: 145.8% margen ($2,728.12 por unidad)
Producto 25: 138.3% margen ($237.84 por unidad)
Producto 24: 133.3% margen ($1,833.39 por unidad)
Producto 15: 132.1% margen ($1,299.42 por unidad)


### 7.3 Distribución de Stock por Categoría

In [20]:
stock_categoria = productos.merge(categorias, left_on='id_categoria', right_on='id', how='left')
stock_por_cat = stock_categoria.groupby('nombre_y')['stock_actual'].sum().sort_values(ascending=False)

fig = px.bar(stock_por_cat.reset_index(), x='nombre_y', y='stock_actual',
             title='Stock Total por Categoría',
             labels={'nombre_y': 'Categoría', 'stock_actual': 'Stock Total'},
             color='stock_actual',
             color_continuous_scale='Blues')
fig.update_layout(showlegend=False, height=500)
fig.show()

## 8. Análisis de Empleados

### 8.1 Ventas por Empleado

In [21]:
ventas_empleados = facturas.merge(empleados, left_on='id_empleado', right_on='id', how='left')
ventas_por_empleado = ventas_empleados.groupby(['nombre', 'apellido', 'cargo']).agg({
    'total': ['sum', 'count', 'mean']
}).reset_index()
ventas_por_empleado.columns = ['Nombre', 'Apellido', 'Cargo', 'Total Vendido', 'Num Ventas', 'Promedio']
ventas_por_empleado['Empleado'] = ventas_por_empleado['Nombre'] + ' ' + ventas_por_empleado['Apellido']
ventas_por_empleado = ventas_por_empleado.sort_values('Total Vendido', ascending=False)

fig = px.bar(ventas_por_empleado, x='Empleado', y='Total Vendido',
             title='Ventas Totales por Empleado',
             labels={'Total Vendido': 'Total Vendido (ARS)', 'Empleado': 'Empleado'},
             color='Total Vendido',
             color_continuous_scale='Teal',
             hover_data=['Cargo', 'Num Ventas', 'Promedio'])
fig.update_layout(showlegend=False, height=500)
fig.show()

### 8.2 Productividad por Departamento

In [22]:
productividad_dept = ventas_empleados.groupby('departamento').agg({
    'total': ['sum', 'count', 'mean']
}).reset_index()
productividad_dept.columns = ['Departamento', 'Total Vendido', 'Num Ventas', 'Promedio por Venta']

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Total por Departamento', 'Promedio por Venta'))

fig.add_trace(go.Bar(x=productividad_dept['Departamento'],
                     y=productividad_dept['Total Vendido'],
                     marker_color='lightcoral', name='Total'),
              row=1, col=1)

fig.add_trace(go.Bar(x=productividad_dept['Departamento'],
                     y=productividad_dept['Promedio por Venta'],
                     marker_color='lightgreen', name='Promedio'),
              row=1, col=2)

fig.update_layout(title_text='Análisis por Departamento', height=500, showlegend=False)
fig.show()

## 9. Análisis Financiero

### 9.1 Estado de Facturas y Pagos

In [23]:
facturas_estado = facturas.groupby('estado').agg({
    'total': ['sum', 'count']
}).reset_index()
facturas_estado.columns = ['Estado', 'Total', 'Cantidad']

print("\n💰 ESTADO FINANCIERO DE FACTURAS:\n")
for _, row in facturas_estado.iterrows():
    print(f"{row['Estado']}: ${row['Total']:,.2f} ({int(row['Cantidad'])} facturas)")

# Facturas vencidas
hoy = pd.Timestamp.now()
facturas_vencidas = facturas[(facturas['fecha_vencimiento'] < hoy) &
                              (facturas['estado'].isin(['Pendiente', 'Vencida']))]

if len(facturas_vencidas) > 0:
    print(f"\n⚠️ ALERTA: {len(facturas_vencidas)} facturas vencidas")
    print(f"   Total en facturas vencidas: ${facturas_vencidas['total'].sum():,.2f}")



💰 ESTADO FINANCIERO DE FACTURAS:

Cancelada: $783,033.51 (28 facturas)
Pagada: $671,016.29 (26 facturas)
Pendiente: $680,040.07 (22 facturas)
Vencida: $724,525.01 (24 facturas)

⚠️ ALERTA: 46 facturas vencidas
   Total en facturas vencidas: $1,404,565.08


### 9.2 Análisis de Pagos

In [24]:
total_pagado = pagos['monto'].sum()
total_facturado = facturas['total'].sum()
porcentaje_cobrado = (total_pagado / total_facturado) * 100

print(f"\n💵 ANÁLISIS DE PAGOS:")
print(f"   Total facturado: ${total_facturado:,.2f}")
print(f"   Total cobrado: ${total_pagado:,.2f}")
print(f"   Porcentaje cobrado: {porcentaje_cobrado:.1f}%")
print(f"   Por cobrar: ${total_facturado - total_pagado:,.2f}")

pagos_metodo = pagos.groupby('metodo_pago')['monto'].sum().sort_values(ascending=False)

fig = px.pie(values=pagos_metodo.values, names=pagos_metodo.index,
             title='Distribución de Pagos por Método',
             hole=0.4)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()


💵 ANÁLISIS DE PAGOS:
   Total facturado: $2,858,614.88
   Total cobrado: $957,117.32
   Porcentaje cobrado: 33.5%
   Por cobrar: $1,901,497.56


### 9.3 Flujo de Caja Mensual

In [25]:
# Ingresos (facturas pagadas)
facturas['mes_num'] = facturas['fecha_emision'].dt.to_period('M')
ingresos = facturas[facturas['estado'] == 'Pagada'].groupby('mes_num')['total'].sum()

# Egresos (compras)
compras['mes_num'] = compras['fecha_orden'].dt.to_period('M')
egresos = compras.groupby('mes_num')['total'].sum()

# Flujo de caja
flujo = pd.DataFrame({'Ingresos': ingresos, 'Egresos': egresos}).fillna(0)
flujo['Flujo Neto'] = flujo['Ingresos'] - flujo['Egresos']
flujo.index = flujo.index.astype(str)

fig = go.Figure()
fig.add_trace(go.Bar(x=flujo.index, y=flujo['Ingresos'], name='Ingresos', marker_color='green'))
fig.add_trace(go.Bar(x=flujo.index, y=flujo['Egresos'], name='Egresos', marker_color='red'))
fig.add_trace(go.Scatter(x=flujo.index, y=flujo['Flujo Neto'], name='Flujo Neto',
                         mode='lines+markers', line=dict(color='blue', width=3)))

fig.update_layout(title='Flujo de Caja Mensual',
                  xaxis_title='Mes',
                  yaxis_title='Monto (ARS)',
                  barmode='group',
                  height=500)
fig.show()

print("\n📊 RESUMEN DE FLUJO DE CAJA:")
print(f"   Total Ingresos: ${flujo['Ingresos'].sum():,.2f}")
print(f"   Total Egresos: ${flujo['Egresos'].sum():,.2f}")
print(f"   Flujo Neto Total: ${flujo['Flujo Neto'].sum():,.2f}")



📊 RESUMEN DE FLUJO DE CAJA:
   Total Ingresos: $671,016.29
   Total Egresos: $2,139,440.34
   Flujo Neto Total: $-1,468,424.05


## 10. Análisis de Proveedores

### 10.1 Compras por Proveedor

In [26]:
compras_prov = compras.merge(proveedores, left_on='id_proveedor', right_on='id', how='left')
compras_por_prov = compras_prov.groupby('nombre_empresa').agg({
    'total': ['sum', 'count', 'mean']
}).reset_index()
compras_por_prov.columns = ['Proveedor', 'Total Comprado', 'Num Compras', 'Promedio']
compras_por_prov = compras_por_prov.sort_values('Total Comprado', ascending=False)

fig = px.bar(compras_por_prov, x='Proveedor', y='Total Comprado',
             title='Compras por Proveedor',
             labels={'Total Comprado': 'Total Comprado (ARS)', 'Proveedor': 'Proveedor'},
             color='Total Comprado',
             color_continuous_scale='Oranges',
             hover_data=['Num Compras', 'Promedio'])
fig.update_layout(showlegend=False, height=500)
fig.show()

print("\n🏭 Top 5 Proveedores:")
for _, row in compras_por_prov.head().iterrows():
    print(f"{row['Proveedor']}: ${row['Total Comprado']:,.2f} en {int(row['Num Compras'])} órdenes")



🏭 Top 5 Proveedores:
TechSupply SA: $514,984.70 en 7 órdenes
AlmacénCentral: $488,171.58 en 8 órdenes
GlobalTrade: $295,370.92 en 5 órdenes
ImportExport SRL: $286,531.37 en 4 órdenes
ComercialSur: $169,527.96 en 6 órdenes


### 10.2 Calificación de Proveedores vs Volumen de Compra

In [27]:
prov_analisis = compras_por_prov.merge(proveedores[['nombre_empresa', 'calificacion']],
                                       left_on='Proveedor', right_on='nombre_empresa', how='left')

fig = px.scatter(prov_analisis, x='calificacion', y='Total Comprado',
                 size='Num Compras', hover_name='Proveedor',
                 title='Calificación vs Volumen de Compras de Proveedores',
                 labels={'calificacion': 'Calificación', 'Total Comprado': 'Total Comprado (ARS)'},
                 color='calificacion',
                 color_continuous_scale='RdYlGn')
fig.update_layout(height=500)
fig.show()

## 11. Dashboard Ejecutivo - KPIs

In [28]:
print("\n" + "="*70)
print("📈 DASHBOARD EJECUTIVO - INDICADORES CLAVE (KPIs)")
print("="*70 + "\n")

# KPIs Principales
total_clientes = len(clientes)
total_productos = len(productos)
total_empleados = len(empleados[empleados['activo'] == 'Sí'])
total_ventas = facturas['total'].sum()
num_facturas = len(facturas)
ticket_promedio = facturas['total'].mean()
productos_stock_bajo = len(productos[productos['stock_actual'] < productos['stock_minimo']])
margen_promedio = productos['margen'].mean()
total_por_cobrar = facturas[facturas['estado'].isin(['Pendiente', 'Vencida'])]['total'].sum()

print(f"👥 CLIENTES")
print(f"   Total de clientes: {total_clientes}")
print(f"   Ciudades atendidas: {clientes['ciudad'].nunique()}")
print(f"   Puntos de fidelidad promedio: {clientes['puntos_fidelidad'].mean():.0f}")

print(f"\n💼 OPERACIONES")
print(f"   Empleados activos: {total_empleados}")
print(f"   Productos en catálogo: {total_productos}")
print(f"   Categorías: {len(categorias)}")
print(f"   Proveedores: {len(proveedores)}")

print(f"\n💰 VENTAS")
print(f"   Total facturado: ${total_ventas:,.2f}")
print(f"   Número de facturas: {num_facturas}")
print(f"   Ticket promedio: ${ticket_promedio:,.2f}")
print(f"   Factura más alta: ${facturas['total'].max():,.2f}")

print(f"\n📦 INVENTARIO")
print(f"   Productos activos: {len(productos[productos['activo'] == 'Sí'])}")
print(f"   ⚠️ Productos con stock bajo: {productos_stock_bajo}")
print(f"   Margen promedio: {margen_promedio:.1f}%")
print(f"   Valor del inventario: ${(productos['precio_venta'] * productos['stock_actual']).sum():,.2f}")

print(f"\n💵 FINANZAS")
print(f"   Total cobrado: ${pagos['monto'].sum():,.2f}")
print(f"   Por cobrar: ${total_por_cobrar:,.2f}")
print(f"   Facturas pendientes: {len(facturas[facturas['estado'] == 'Pendiente'])}")
print(f"   Facturas vencidas: {len(facturas[facturas['estado'] == 'Vencida'])}")

print(f"\n🏭 COMPRAS")
print(f"   Total en compras: ${compras['total'].sum():,.2f}")
print(f"   Número de órdenes: {len(compras)}")
print(f"   Compra promedio: ${compras['total'].mean():,.2f}")

print("\n" + "="*70)



📈 DASHBOARD EJECUTIVO - INDICADORES CLAVE (KPIs)

👥 CLIENTES
   Total de clientes: 50
   Ciudades atendidas: 12
   Puntos de fidelidad promedio: 233

💼 OPERACIONES
   Empleados activos: 12
   Productos en catálogo: 30
   Categorías: 7
   Proveedores: 11

💰 VENTAS
   Total facturado: $2,858,614.88
   Número de facturas: 100
   Ticket promedio: $28,586.15
   Factura más alta: $60,493.61

📦 INVENTARIO
   Productos activos: 25
   ⚠️ Productos con stock bajo: 2
   Margen promedio: 87.4%
   Valor del inventario: $2,214,884.53

💵 FINANZAS
   Total cobrado: $957,117.32
   Por cobrar: $1,404,565.08
   Facturas pendientes: 22
   Facturas vencidas: 24

🏭 COMPRAS
   Total en compras: $2,139,440.34
   Número de órdenes: 40
   Compra promedio: $53,486.01



## 12. Visualización de KPIs en Dashboard

In [29]:
# Crear un dashboard visual con los KPIs principales
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=('Ventas por Mes', 'Top 5 Productos', 'Métodos de Pago',
                    'Stock por Categoría', 'Ventas por Empleado', 'Estado de Facturas'),
    specs=[[{'type': 'scatter'}, {'type': 'bar'}, {'type': 'pie'}],
           [{'type': 'bar'}, {'type': 'bar'}, {'type': 'pie'}]]
)

# 1. Ventas por mes
ventas_mes_simple = facturas.groupby(facturas['fecha_emision'].dt.to_period('M'))['total'].sum()
fig.add_trace(go.Scatter(x=[str(x) for x in ventas_mes_simple.index],
                         y=ventas_mes_simple.values,
                         mode='lines+markers',
                         name='Ventas',
                         line=dict(color='blue', width=2)),
              row=1, col=1)

# 2. Top 5 productos
top5_prod = ventas_productos.groupby('nombre')['cantidad'].sum().nlargest(5)
fig.add_trace(go.Bar(x=top5_prod.values, y=top5_prod.index,
                     orientation='h',
                     marker_color='lightblue',
                     name='Productos'),
              row=1, col=2)

# 3. Métodos de pago
pago_dist = facturas['tipo_pago'].value_counts()
fig.add_trace(go.Pie(labels=pago_dist.index, values=pago_dist.values,
                     name='Pago'),
              row=1, col=3)

# 4. Stock por categoría
stock_cat = stock_categoria.groupby('nombre_y')['stock_actual'].sum().nlargest(5)
fig.add_trace(go.Bar(x=stock_cat.index, y=stock_cat.values,
                     marker_color='lightgreen',
                     name='Stock'),
              row=2, col=1)

# 5. Top 5 empleados
top5_emp = ventas_por_empleado.head(5)
fig.add_trace(go.Bar(x=top5_emp['Empleado'], y=top5_emp['Total Vendido'],
                     marker_color='lightcoral',
                     name='Empleados'),
              row=2, col=2)

# 6. Estado de facturas
estado_dist = facturas['estado'].value_counts()
fig.add_trace(go.Pie(labels=estado_dist.index, values=estado_dist.values,
                     name='Estado'),
              row=2, col=3)

fig.update_layout(height=800, showlegend=False, title_text="Dashboard Ejecutivo - Resumen Visual")
fig.show()

## 13. Alertas y Recomendaciones

In [30]:
print("\n" + "="*70)
print("⚠️ ALERTAS Y RECOMENDACIONES")
print("="*70 + "\n")

alertas = []
recomendaciones = []

# Alerta 1: Stock bajo
if productos_stock_bajo > 0:
    alertas.append(f"🔴 {productos_stock_bajo} productos con stock por debajo del mínimo")
    recomendaciones.append("→ Realizar pedido urgente a proveedores para productos críticos")

# Alerta 2: Facturas vencidas
facturas_venc = len(facturas_vencidas) if len(facturas_vencidas) > 0 else 0
if facturas_venc > 0:
    alertas.append(f"🔴 {facturas_venc} facturas vencidas sin pagar (${facturas_vencidas['total'].sum():,.2f})")
    recomendaciones.append("→ Contactar clientes con facturas vencidas para gestionar cobros")

# Alerta 3: Productos sin ventas
productos_vendidos = detalle_facturas['id_producto'].unique()
productos_sin_venta = productos[~productos['id'].isin(productos_vendidos)]
if len(productos_sin_venta) > 0:
    alertas.append(f"🟡 {len(productos_sin_venta)} productos sin ventas registradas")
    recomendaciones.append("→ Evaluar promociones o descontinuar productos sin movimiento")

# Alerta 4: Clientes inactivos (sin compras recientes)
ultima_compra_cliente = facturas.groupby('id_cliente')['fecha_emision'].max()
clientes_inactivos = ultima_compra_cliente[ultima_compra_cliente < (pd.Timestamp.now() - pd.Timedelta(days=90))]
if len(clientes_inactivos) > 0:
    alertas.append(f"🟡 {len(clientes_inactivos)} clientes sin compras en los últimos 90 días")
    recomendaciones.append("→ Implementar campaña de reactivación para clientes inactivos")

# Alerta 5: Margen bajo en productos
productos_margen_bajo = productos[productos['margen'] < 20]
if len(productos_margen_bajo) > 0:
    alertas.append(f"🟡 {len(productos_margen_bajo)} productos con margen inferior al 20%")
    recomendaciones.append("→ Revisar precios o negociar mejores condiciones con proveedores")

# Mostrar alertas
if alertas:
    print("🚨 ALERTAS DETECTADAS:\n")
    for alerta in alertas:
        print(f"   {alerta}")
else:
    print("✅ No se detectaron alertas críticas")

# Mostrar recomendaciones
if recomendaciones:
    print("\n💡 RECOMENDACIONES:\n")
    for i, rec in enumerate(recomendaciones, 1):
        print(f"   {i}. {rec}")

print("\n" + "="*70)


⚠️ ALERTAS Y RECOMENDACIONES

🚨 ALERTAS DETECTADAS:

   🔴 2 productos con stock por debajo del mínimo
   🔴 46 facturas vencidas sin pagar ($1,404,565.08)
   🟡 42 clientes sin compras en los últimos 90 días

💡 RECOMENDACIONES:

   1. → Realizar pedido urgente a proveedores para productos críticos
   2. → Contactar clientes con facturas vencidas para gestionar cobros
   3. → Implementar campaña de reactivación para clientes inactivos



## 14. Análisis Avanzado - Insights Adicionales

### 14.1 Análisis de Rentabilidad por Producto

In [31]:
# Calcular rentabilidad real considerando ventas
rentabilidad_productos = detalle_facturas.merge(productos[['id', 'nombre', 'precio_compra', 'precio_venta', 'margen']],
                                                 left_on='id_producto', right_on='id', how='left')

rentabilidad_productos['ganancia_total'] = (rentabilidad_productos['precio_venta'] -
                                             rentabilidad_productos['precio_compra']) * rentabilidad_productos['cantidad']

rentabilidad_por_producto = rentabilidad_productos.groupby('nombre').agg({
    'cantidad': 'sum',
    'ganancia_total': 'sum',
    'margen': 'first'
}).sort_values('ganancia_total', ascending=False).head(10)

fig = px.bar(rentabilidad_por_producto.reset_index(),
             x='nombre', y='ganancia_total',
             title='Top 10 Productos por Ganancia Total Generada',
             labels={'ganancia_total': 'Ganancia Total (ARS)', 'nombre': 'Producto'},
             color='ganancia_total',
             color_continuous_scale='Greens',
             hover_data=['cantidad', 'margen'])
fig.update_layout(showlegend=False, height=500)
fig.show()

print("\n💰 Top 5 Productos Más Rentables (Ganancia Total):")
for i, (nombre, data) in enumerate(rentabilidad_por_producto.head().iterrows(), 1):
    print(f"{i}. {nombre}")
    print(f"   Ganancia: ${data['ganancia_total']:,.2f}")
    print(f"   Unidades vendidas: {int(data['cantidad'])}")
    print(f"   Margen: {data['margen']:.1f}%\n")


💰 Top 5 Productos Más Rentables (Ganancia Total):
1. Producto 6
   Ganancia: $176,379.30
   Unidades vendidas: 90
   Margen: 130.3%

2. Producto 24
   Ganancia: $139,337.64
   Unidades vendidas: 76
   Margen: 133.3%

3. Producto 11
   Ganancia: $131,315.38
   Unidades vendidas: 79
   Margen: 88.9%

4. Producto 30
   Ganancia: $113,766.42
   Unidades vendidas: 87
   Margen: 86.6%

5. Producto 29
   Ganancia: $109,124.80
   Unidades vendidas: 40
   Margen: 145.8%



### 14.2 Análisis de Estacionalidad en Ventas

In [32]:
# Ventas por día de la semana
facturas['dia_semana'] = facturas['fecha_emision'].dt.day_name()
ventas_por_dia = facturas.groupby('dia_semana')['total'].agg(['sum', 'count', 'mean'])

dias_orden = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dias_es = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
ventas_por_dia = ventas_por_dia.reindex(dias_orden)
ventas_por_dia.index = dias_es

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Ventas Totales por Día', 'Cantidad de Transacciones'))

fig.add_trace(go.Bar(x=ventas_por_dia.index, y=ventas_por_dia['sum'],
                     marker_color='lightblue', name='Total'),
              row=1, col=1)

fig.add_trace(go.Bar(x=ventas_por_dia.index, y=ventas_por_dia['count'],
                     marker_color='lightcoral', name='Cantidad'),
              row=1, col=2)

fig.update_layout(title_text='Análisis de Ventas por Día de la Semana', height=500, showlegend=False)
fig.show()

print("\n📅 Día con más ventas:", ventas_por_dia['sum'].idxmax(),
      f"(${ventas_por_dia['sum'].max():,.2f})")
print(f"📅 Día con más transacciones:", ventas_por_dia['count'].idxmax(),
      f"({int(ventas_por_dia['count'].max())} transacciones)")


📅 Día con más ventas: Martes ($652,687.06)
📅 Día con más transacciones: Sábado (21 transacciones)


 ### 14.3 Análisis RFM Simplificado (Recency, Frequency, Monetary)

In [33]:
# Calcular métricas RFM para clientes
fecha_referencia = facturas['fecha_emision'].max()

rfm = facturas.groupby('id_cliente').agg({
    'fecha_emision': lambda x: (fecha_referencia - x.max()).days,  # Recency
    'id': 'count',  # Frequency
    'total': 'sum'  # Monetary
}).reset_index()

rfm.columns = ['id_cliente', 'Recency', 'Frequency', 'Monetary']

# Añadir nombres de clientes
rfm = rfm.merge(clientes[['id', 'nombre', 'apellido']], left_on='id_cliente', right_on='id', how='left')
rfm['Cliente'] = rfm['nombre'] + ' ' + rfm['apellido']

# Crear scores (1-5) para cada métrica
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1], duplicates='drop')
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5], duplicates='drop')
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5], duplicates='drop')

rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

# Segmentar clientes
def segmentar_cliente(row):
    r, f, m = int(row['R_Score']), int(row['F_Score']), int(row['M_Score'])
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4:
        return 'Promising'
    elif f >= 4:
        return 'Potential Loyalist'
    elif r <= 2:
        return 'At Risk'
    elif r <= 2 and f <= 2:
        return 'Hibernating'
    else:
        return 'Need Attention'

rfm['Segmento'] = rfm.apply(segmentar_cliente, axis=1)

# Visualizar segmentos
segmentos_dist = rfm['Segmento'].value_counts()

fig = px.pie(values=segmentos_dist.values, names=segmentos_dist.index,
             title='Segmentación de Clientes (Análisis RFM)',
             hole=0.4,
             color_discrete_sequence=px.colors.qualitative.Set3)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

print("\n🎯 SEGMENTACIÓN DE CLIENTES:\n")
for segmento, count in segmentos_dist.items():
    clientes_seg = rfm[rfm['Segmento'] == segmento]
    print(f"{segmento}: {count} clientes")
    print(f"  Valor promedio: ${clientes_seg['Monetary'].mean():,.2f}")
    print(f"  Compras promedio: {clientes_seg['Frequency'].mean():.1f}\n")

# Top 10 clientes por RFM
print("🏆 Top 10 Mejores Clientes (RFM):\n")
top_rfm = rfm.nlargest(10, 'Monetary')[['Cliente', 'Recency', 'Frequency', 'Monetary', 'Segmento']]
for i, row in top_rfm.iterrows():
    print(f"{row['Cliente']} - {row['Segmento']}")
    print(f"  Última compra: hace {int(row['Recency'])} días")
    print(f"  Total compras: {int(row['Frequency'])}")
    print(f"  Valor total: ${row['Monetary']:,.2f}\n")


🎯 SEGMENTACIÓN DE CLIENTES:

Loyal Customers: 11 clientes
  Valor promedio: $62,867.78
  Compras promedio: 2.7

At Risk: 11 clientes
  Valor promedio: $46,772.72
  Compras promedio: 1.5

Potential Loyalist: 6 clientes
  Valor promedio: $94,627.55
  Compras promedio: 3.3

Need Attention: 5 clientes
  Valor promedio: $49,742.91
  Compras promedio: 1.4

Champions: 5 clientes
  Valor promedio: $135,740.01
  Compras promedio: 3.8

Promising: 4 clientes
  Valor promedio: $39,347.35
  Compras promedio: 1.8

🏆 Top 10 Mejores Clientes (RFM):

María Flores - Champions
  Última compra: hace 0 días
  Total compras: 5
  Valor total: $186,631.84

Nicolás Castro - Champions
  Última compra: hace 39 días
  Total compras: 3
  Valor total: $141,335.61

Martín Martínez - Champions
  Última compra: hace 16 días
  Total compras: 3
  Valor total: $132,928.48

Laura Romero - Loyal Customers
  Última compra: hace 86 días
  Total compras: 4
  Valor total: $119,724.83

Carlos Vargas - Champions
  Última compra

### 14.4 Análisis de Correlaciones

In [34]:
# Crear matriz de correlación con variables numéricas relevantes
facturas_num = facturas[['subtotal', 'descuento', 'impuestos', 'total']].copy()
productos_num = productos[['precio_compra', 'precio_venta', 'stock_actual', 'margen']].copy()

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Correlación en Facturas', 'Correlación en Productos'),
                    specs=[[{'type': 'heatmap'}, {'type': 'heatmap'}]])

# Heatmap facturas
corr_facturas = facturas_num.corr()
fig.add_trace(go.Heatmap(z=corr_facturas.values,
                         x=corr_facturas.columns,
                         y=corr_facturas.columns,
                         colorscale='RdBu',
                         zmid=0,
                         text=corr_facturas.values.round(2),
                         texttemplate='%{text}',
                         textfont={"size": 10}),
              row=1, col=1)

# Heatmap productos
corr_productos = productos_num.corr()
fig.add_trace(go.Heatmap(z=corr_productos.values,
                         x=corr_productos.columns,
                         y=corr_productos.columns,
                         colorscale='RdBu',
                         zmid=0,
                         text=corr_productos.values.round(2),
                         texttemplate='%{text}',
                         textfont={"size": 10}),
              row=1, col=2)

fig.update_layout(title_text='Matriz de Correlaciones', height=500)
fig.show()

## 15. Resumen Ejecutivo Final

In [35]:
print("\n" + "="*70)
print("📋 RESUMEN EJECUTIVO FINAL")
print("="*70 + "\n")

print("🎯 HALLAZGOS PRINCIPALES:\n")

# 1. Rendimiento de ventas
crecimiento_ventas = ((ventas_mes.iloc[-1]['total'] - ventas_mes.iloc[0]['total']) /
                      ventas_mes.iloc[0]['total'] * 100)
print(f"1. VENTAS:")
print(f"   • Total facturado: ${total_ventas:,.2f}")
print(f"   • Tendencia: {'Crecimiento' if crecimiento_ventas > 0 else 'Decrecimiento'} del {abs(crecimiento_ventas):.1f}%")
print(f"   • Ticket promedio: ${ticket_promedio:,.2f}")
print(f"   • Mejor mes: {ventas_mes.loc[ventas_mes['total'].idxmax(), 'mes']}")

# 2. Clientes
print(f"\n2. CLIENTES:")
print(f"   • Total: {total_clientes} clientes en {clientes['ciudad'].nunique()} ciudades")
print(f"   • Segmento principal: {rfm['Segmento'].mode()[0]}")
print(f"   • Método de pago preferido: {facturas['tipo_pago'].mode()[0]}")

# 3. Productos
print(f"\n3. PRODUCTOS:")
print(f"   • Catálogo: {total_productos} productos en {len(categorias)} categorías")
print(f"   • Margen promedio: {margen_promedio:.1f}%")
print(f"   • ⚠️ Stock crítico: {productos_stock_bajo} productos")
print(f"   • Producto estrella: {top_productos.index[0]}")

# 4. Operaciones
print(f"\n4. OPERACIONES:")
print(f"   • Empleados: {total_empleados} activos")
print(f"   • Mejor vendedor: {ventas_por_empleado.iloc[0]['Empleado']}")
print(f"   • Proveedores: {len(proveedores)}")
print(f"   • Principal proveedor: {compras_por_prov.iloc[0]['Proveedor']}")

# 5. Finanzas
cobrado_porcentaje = (pagos['monto'].sum() / total_ventas * 100)
print(f"\n5. FINANZAS:")
print(f"   • Total cobrado: {cobrado_porcentaje:.1f}% (${pagos['monto'].sum():,.2f})")
print(f"   • Por cobrar: ${total_por_cobrar:,.2f}")
print(f"   • Flujo neto: ${flujo['Flujo Neto'].sum():,.2f}")
print(f"   • ⚠️ Facturas vencidas: {facturas_venc}")

print("\n" + "="*70)

print("\n💡 RECOMENDACIONES ESTRATÉGICAS:\n")
print("1. Enfocarse en productos de alto margen y alta rotación")
print("2. Implementar programa de fidelización para 'Champions'")
print("3. Realizar seguimiento intensivo de cuentas por cobrar")
print("4. Reabastecer urgentemente productos con stock crítico")
print("5. Analizar y reactivar clientes en segmento 'At Risk'")
print("6. Optimizar precios de productos con margen bajo")
print("7. Fortalecer relación con proveedores de mejor calificación")

print("\n" + "="*70)
print("✅ ANÁLISIS COMPLETADO")
print("="*70 + "\n")


📋 RESUMEN EJECUTIVO FINAL

🎯 HALLAZGOS PRINCIPALES:

1. VENTAS:
   • Total facturado: $2,858,614.88
   • Tendencia: Crecimiento del 50.9%
   • Ticket promedio: $28,586.15
   • Mejor mes: 2024-06

2. CLIENTES:
   • Total: 50 clientes en 12 ciudades
   • Segmento principal: At Risk
   • Método de pago preferido: Tarjeta Débito

3. PRODUCTOS:
   • Catálogo: 30 productos en 7 categorías
   • Margen promedio: 87.4%
   • ⚠️ Stock crítico: 2 productos
   • Producto estrella: Producto 27

4. OPERACIONES:
   • Empleados: 12 activos
   • Mejor vendedor: Valentina Vargas
   • Proveedores: 11
   • Principal proveedor: TechSupply SA

5. FINANZAS:
   • Total cobrado: 33.5% ($957,117.32)
   • Por cobrar: $1,404,565.08
   • Flujo neto: $-1,468,424.05
   • ⚠️ Facturas vencidas: 46


💡 RECOMENDACIONES ESTRATÉGICAS:

1. Enfocarse en productos de alto margen y alta rotación
2. Implementar programa de fidelización para 'Champions'
3. Realizar seguimiento intensivo de cuentas por cobrar
4. Reabastecer urge

## 16. Exportar Reportes

In [36]:
print("\n¿Deseas exportar los resultados del análisis?")
print("Los siguientes archivos pueden ser descargados:\n")

# Crear resumen en CSV
resumen_kpis = pd.DataFrame({
    'KPI': ['Total Clientes', 'Total Ventas', 'Ticket Promedio', 'Productos',
            'Stock Bajo', 'Margen Promedio', 'Por Cobrar', 'Facturas Vencidas'],
    'Valor': [total_clientes, f'${total_ventas:,.2f}', f'${ticket_promedio:,.2f}',
              total_productos, productos_stock_bajo, f'{margen_promedio:.1f}%',
              f'${total_por_cobrar:,.2f}', facturas_venc]
})

resumen_kpis.to_csv('resumen_kpis.csv', index=False)
top_productos.to_csv('top_productos.csv')
top_clientes.to_frame().to_csv('top_clientes.csv')
ventas_por_empleado.to_csv('ventas_empleados.csv', index=False)
rfm.to_csv('segmentacion_clientes_rfm.csv', index=False)
stock_bajo.to_csv('alerta_stock_bajo.csv', index=False)

print("✓ resumen_kpis.csv")
print("✓ top_productos.csv")
print("✓ top_clientes.csv")
print("✓ ventas_empleados.csv")
print("✓ segmentacion_clientes_rfm.csv")
print("✓ alerta_stock_bajo.csv")

print("\n📥 Archivos listos para descargar desde el panel de archivos de Colab")



¿Deseas exportar los resultados del análisis?
Los siguientes archivos pueden ser descargados:

✓ resumen_kpis.csv
✓ top_productos.csv
✓ top_clientes.csv
✓ ventas_empleados.csv
✓ segmentacion_clientes_rfm.csv
✓ alerta_stock_bajo.csv

📥 Archivos listos para descargar desde el panel de archivos de Colab
